<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_09-3_RPG_LangGraph_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 3 | LangGraph로 AI 전투 에이전트 만들기

**Colab 실습 · API 키 불필요**

> **핵심 질문:** AI가 게임 상태를 확인하고(`observe`), 행동을 정하고(`decide`), MCP Tool로 실행한 후(`act`), 결과에 따라 다시 행동하거나 끝내려면(`judge`) 어떤 구조가 필요한가?

### 학습 목표
1. `StateGraph`, `State`, `Node`, `Edge`의 역할을 설명한다.
2. `add_conditional_edges()`로 반복과 종료를 구현한다.
3. `Client(mcp)`에서 MCP Tool을 호출해 게임 상태를 변경한다.
4. `max_turns`와 `recursion_limit`의 차이를 확인한다.
5. **규칙 기반 정책**과 **LLM 기반 Agent**를 구별한다.

**정확한 구분:** 이 기본 실습은 LLM 없이 규칙으로 행동을 선택하는 **LangGraph 전투 워크플로**입니다. 따라서 'AI 에이전트의 실행 구조'를 학습하지만, 모델이 스스로 행동을 선택하는 실제 LLM Agent는 아닙니다. 마지막 선택 과제에서 LLM 연동 방향을 제시합니다.

## 0. 실행 준비

- **새 Colab 노트북 하나만으로 독립 실행**됩니다. 실습 2 파일을 가져오지 않아도 됩니다.
- `langgraph==1.2.20`, `mcp==2.3.0`을 고정해 강의자료의 버전과 맞춥니다.
- 첫 셀의 패키지 설치에는 인터넷이 필요하지만 **OpenAI API 키는 필요 없습니다**.
- 코드에는 학습 편의를 위해 **실행문마다 한글 주석**을 달았습니다.
- **게임 규칙 변화:** 실습 2와 HP(100), 슬라임 HP(60), 공격 피해(20), 포션(2개)은 같고, 의사결정 실습을 위해 슬라임 반격 피해(18)를 추가했습니다.

> **검증 범위:** 제작 환경의 외부 네트워크가 차단되어 MCP/LangGraph 라이브러리의 전체 통합 실행은 수행하지 못했습니다. Python 구문 검사 및 독립 전투 로직 검증은 수행하며, 통합 검증 셀을 별도로 제공합니다.

In [1]:
# Colab에 MCP Python SDK와 LangGraph를 지정 버전으로 설치합니다.
%pip -q install "mcp==2.3.0" "langgraph==1.2.20"

In [2]:
from importlib.metadata import version  # 설치된 라이브러리의 버전 확인에 사용합니다.
from dataclasses import dataclass, asdict  # 게임 상태를 간단한 객체로 정의하고 딕셔너리로 바꿉니다.
from typing import Any, Annotated, TypedDict  # 도구 결과와 LangGraph State의 자료형을 명시합니다.
import operator  # LangGraph에서 이벤트 목록을 누적할 때 사용할 함수를 제공합니다.
import json  # MCP Tool 결과를 화면에 보기 쉽게 출력합니다.
from mcp.server import MCPServer  # 게임 함수를 등록할 MCP 서버 클래스입니다.
from mcp import Client  # 서버를 메모리 내에서 호출할 MCP 클라이언트입니다.
from langgraph.graph import StateGraph, START, END  # 상태 그래프 생성과 시작/종료 노드 정의에 사용합니다.
print("MCP 버전:", version("mcp"))  # MCP 패키지 버전을 확인합니다.
print("LangGraph 버전:", version("langgraph"))  # LangGraph 패키지 버전을 확인합니다.
assert version("mcp") == "2.3.0"  # 강의에서 정한 MCP 버전과 일치하는지 확인합니다.
assert version("langgraph") == "1.2.20"  # 강의에서 정한 LangGraph 버전과 일치하는지 확인합니다.

MCP 버전: 2.3.0
LangGraph 버전: 1.2.20


## 1. RPG 게임 엔진 구현 (상태의 실제 소유자)

**중요:** HP·포션·턴의 원본 데이터는 **게임 엔진**이 관리합니다. LangGraph의 `BattleState`는 그 상태의 관측값 및 에이전트 진행 기록을 담습니다. 두 곳을 동시에 원본으로 관리하지 않도록 분리합니다.

| 항목 | 시작값 | 규칙 |
|---|---:|---|
| 플레이어 HP | 100 | 0 이하면 패배 |
| 슬라임 HP | 60 | 0 이하면 승리 |
| 공격 | 20 피해 | 몬스터가 살아 있으면 18 반격 |
| 방어 | 반격 피해 9 | 공격 대신 방어 |
| 회복 | 포션 1개로 HP 30 회복 | 최대 100; 몬스터 반격 있음 |
| 최대 턴 | 10 | 이후 `timeout` 처리 |

**실습을 위해 전투를 결정론적으로 설계**했습니다. 난수에 의한 결과 변동이 없어 서로 다른 전략을 정확히 비교할 수 있습니다.

In [3]:
@dataclass  # 게임 데이터를 자동 초기화할 수 있게 dataclass로 정의합니다.
class RPGGame:  # MCP 서버 안에서 관리하는 하나의 RPG 전투 상태입니다.
    player_hp: int = 100  # 플레이어의 최초 HP입니다.
    enemy_hp: int = 60  # 슬라임의 최초 HP입니다.
    potions: int = 2  # 사용할 수 있는 포션의 최초 개수입니다.
    turn: int = 0  # 실제로 행동을 실행한 턴 수입니다.
    outcome: str = "playing"  # 현재 전투 상태: playing/win/lose/timeout 중 하나입니다.

GAME = RPGGame()  # 서버가 실제 게임 원본을 보관하는 전역 객체입니다.
MAX_HP = 100  # 회복 후에도 넘을 수 없는 최대 체력입니다.
ATTACK_DAMAGE = 20  # 플레이어 공격의 고정 피해량입니다.
COUNTER_DAMAGE = 18  # 슬라임이 살아 있을 때 주는 고정 반격 피해입니다.
MAX_TURNS = 10  # 종료 규칙에 사용할 최대 행동 횟수입니다.

def reset_logic(player_hp: int = 100) -> dict[str, Any]:  # 테스트 시작 시 게임을 초기화하는 함수입니다.
    global GAME  # 새 게임 객체를 전역 상태에 다시 저장합니다.
    if not isinstance(player_hp, int) or isinstance(player_hp, bool) or not 1 <= player_hp <= MAX_HP:  # HP 입력의 자료형과 범위를 검증합니다.
        return {"ok": False, "error_code": "INVALID_HP"}  # 잘못된 HP는 게임을 변경하지 않고 거부합니다.
    GAME = RPGGame(player_hp=player_hp)  # 전달받은 HP로 새로운 게임을 시작합니다.
    return {"ok": True, "state": asdict(GAME)}  # 초기화한 게임 상태를 구조화해서 반환합니다.

def status_logic() -> dict[str, Any]:  # 현재 게임을 읽기만 하는 함수입니다.
    return {"ok": True, "state": asdict(GAME)}  # 호출해도 HP와 턴은 바뀌지 않습니다.

def perform_logic(action: str, target: str = "slime") -> dict[str, Any]:  # 실제 전투 동작을 한곳에서 처리하는 함수입니다.
    if GAME.outcome != "playing":  # 전투가 이미 끝났는지 먼저 확인합니다.
        return {"ok": False, "error_code": "GAME_FINISHED", "state": asdict(GAME)}  # 종료된 게임에서의 재실행을 막습니다.
    if action not in {"attack", "heal", "defend"}:  # 구현된 행동인지 검사합니다.
        return {"ok": False, "error_code": "UNKNOWN_ACTION", "state": asdict(GAME)}  # 알 수 없는 행동은 거부합니다.
    if action == "attack" and target != "slime":  # 공격 대상이 허용된 몬스터인지 검사합니다.
        return {"ok": False, "error_code": "INVALID_TARGET", "state": asdict(GAME)}  # 잘못된 대상에 대해서는 턴을 소비하지 않습니다.
    if action == "heal" and GAME.potions <= 0:  # 회복 전에 남은 포션 수를 검사합니다.
        return {"ok": False, "error_code": "NO_POTIONS", "state": asdict(GAME)}  # 포션이 없으면 아무것도 변경하지 않습니다.
    if action == "heal" and GAME.player_hp >= MAX_HP:  # 최대 HP일 때 포션 사용을 방지합니다.
        return {"ok": False, "error_code": "HP_FULL", "state": asdict(GAME)}  # 불필요한 소비를 막고 정상적인 업무 오류로 돌려줍니다.
    GAME.turn += 1  # 정상적인 행동 요청일 때에만 턴을 한 번 증가시킵니다.
    if action == "attack":  # 공격 행동을 선택했는지 확인합니다.
        GAME.enemy_hp = max(0, GAME.enemy_hp - ATTACK_DAMAGE)  # 슬라임 HP를 감소시키되 음수가 되지 않게 합니다.
    elif action == "heal":  # 회복 행동을 선택했는지 확인합니다.
        GAME.potions -= 1  # 포션을 하나 소비합니다.
        GAME.player_hp = min(MAX_HP, GAME.player_hp + 30)  # 플레이어 체력을 최대 30 회복합니다.
    damage_taken = 0  # 이번 턴에 실제로 받은 피해량을 초기화합니다.
    if GAME.enemy_hp > 0:  # 몬스터가 살아 있을 때에만 반격합니다.
        damage_taken = 9 if action == "defend" else COUNTER_DAMAGE  # 방어 시 반격 피해를 절반으로 줄입니다.
        GAME.player_hp = max(0, GAME.player_hp - damage_taken)  # 플레이어 HP에서 반격 피해를 차감합니다.
    if GAME.enemy_hp == 0:  # 몬스터 HP가 0인 경우를 검사합니다.
        GAME.outcome = "win"  # 승리로 전투 상태를 변경합니다.
    elif GAME.player_hp == 0:  # 플레이어 HP가 0인 경우를 검사합니다.
        GAME.outcome = "lose"  # 패배로 전투 상태를 변경합니다.
    elif GAME.turn >= MAX_TURNS:  # 승패가 나지 않았어도 최대 턴을 넘겼는지 검사합니다.
        GAME.outcome = "timeout"  # 끝없는 전투를 방지하기 위해 강제 종료합니다.
    return {"ok": True, "action": action, "damage_taken": damage_taken, "state": asdict(GAME)}  # 행동의 결과와 갱신된 상태를 반환합니다.

## 2. MCP Tool 등록

실습 2와 동일하게 `@mcp.tool()`로 함수를 공개합니다. **게임 데이터는 Python 서버 함수가 변경하고**, LangGraph는 MCP Client로 서버 함수 실행을 요청합니다.

In [4]:
mcp = MCPServer("RPG Battle Server")  # 새로운 MCP 게임 서버를 생성합니다.

@mcp.tool()  # 현재 게임 상태를 조회하는 함수를 MCP Tool로 등록합니다.
def get_game_status() -> dict[str, Any]:  # 입력 없이 게임 상태만 돌려주는 읽기 도구입니다.
    """현재 HP, 포션 수, 턴 수, 전투 결과를 조회한다."""  # 모델/클라이언트에 제공할 도구 설명입니다.
    return status_logic()  # 원본 상태를 변경하지 않고 반환합니다.

@mcp.tool()  # 슬라임을 공격하는 함수를 MCP Tool로 등록합니다.
def attack_enemy(target: str) -> dict[str, Any]:  # 공격하려는 몬스터 이름을 입력받습니다.
    """현재 전투 중인 slime을 공격해 HP를 20 낮춘다."""  # 공격 도구의 기능을 명확히 설명합니다.
    return perform_logic("attack", target)  # 전투 엔진에 공격 처리를 맡깁니다.

@mcp.tool()  # 플레이어를 회복시키는 함수를 MCP Tool로 등록합니다.
def heal_player() -> dict[str, Any]:  # 회복은 기본 포션을 사용하므로 인자가 필요 없습니다.
    """포션 1개를 사용하여 최대 30 HP 회복한다."""  # 회복 도구의 규칙을 설명합니다.
    return perform_logic("heal")  # 전투 엔진에 회복 처리를 맡깁니다.

@mcp.tool()  # 몬스터의 피해를 줄이는 함수를 MCP Tool로 등록합니다.
def defend_player() -> dict[str, Any]:  # 현재 턴에 방어 행동을 수행합니다.
    """이번 턴 슬라임 반격 피해를 18에서 9로 줄인다."""  # 방어 도구의 효과를 설명합니다.
    return perform_logic("defend")  # 전투 엔진에 방어 처리를 맡깁니다.

print("등록한 MCP Tool:", ["get_game_status", "attack_enemy", "heal_player", "defend_player"])  # 학습자가 공개한 네 도구 이름을 확인합니다.

등록한 MCP Tool: ['get_game_status', 'attack_enemy', 'heal_player', 'defend_player']


## 3. MCP Client 연결 테스트

MCP 호출이 성공했는지 `is_error`와 `structured_content`를 구분해서 확인합니다. `ok: False`는 서버가 정상 처리한 **게임 업무 오류**일 수 있습니다.

In [5]:
async with Client(mcp) as client:  # 외부 포트 없이 In-memory MCP Client를 실행합니다.
    tools = await client.list_tools()  # 서버에 등록된 Tool 목록을 조회합니다.
    tool_names = [tool.name for tool in tools.tools]  # 도구 이름만 뽑아 목록으로 만듭니다.
    print("MCP Tool 목록:", tool_names)  # 등록된 도구 이름을 화면에 출력합니다.
    assert {"get_game_status", "attack_enemy", "heal_player", "defend_player"} <= set(tool_names)  # 필수 도구가 모두 등록됐는지 검증합니다.
    result = await client.call_tool("get_game_status", {})  # 읽기 전용 상태 조회 도구를 MCP로 호출합니다.
    assert not result.is_error  # MCP 요청 자체가 오류 없이 완료됐는지 확인합니다.
    assert result.structured_content["state"]["enemy_hp"] == 60  # 최초 몬스터 HP가 60인지 확인합니다.
    print("초기 상태:", json.dumps(result.structured_content, ensure_ascii=False))  # 구조화된 결과를 출력합니다.

MCP Tool 목록: ['get_game_status', 'attack_enemy', 'heal_player', 'defend_player']
초기 상태: {"ok": true, "state": {"player_hp": 100, "enemy_hp": 60, "potions": 2, "turn": 0, "outcome": "playing"}}


## 4. LangGraph State 정의

- **게임 원본 상태:** `GAME` (MCP 서버 내부)
- **LangGraph State:** 현재 관측값과 선택한 행동, MCP 결과, 단계 수, 실행 로그를 운반하는 기록
- `events: Annotated[list[str], operator.add]`는 여러 노드의 **로그 조각을 누적**하도록 하는 reducer입니다. **이 예제는 순차 실행**이므로 충돌 회피 목적이 아니라 기록 누적을 보여주는 용도로 사용합니다.

In [6]:
class BattleState(TypedDict):  # LangGraph가 노드 사이에 전달할 데이터의 형식을 정의합니다.
    observation: dict[str, Any]  # MCP 서버에서 읽은 최신 게임 상태를 보관합니다.
    action: str  # 판단 노드가 선택한 다음 행동 이름을 기록합니다.
    last_result: dict[str, Any]  # 마지막 MCP Tool 호출의 구조화 결과를 기록합니다.
    steps: int  # 그래프의 행동 실행 횟수를 기록합니다.
    route: str  # 종료 또는 반복 분기 판단 결과를 기록합니다.
    strategy: str  # cautious 또는 aggressive 전략 이름을 기록합니다.
    events: Annotated[list[str], operator.add]  # 노드가 돌려준 로그 목록을 덧붙이는 reducer입니다.

def choose_action(observation: dict[str, Any], strategy: str = "cautious") -> str:  # 규칙으로 행동을 정하는 함수를 만듭니다.
    hp = observation["player_hp"]  # 현재 플레이어 HP를 추출합니다.
    potions = observation["potions"]  # 사용 가능한 포션 수를 추출합니다.
    if strategy == "cautious" and hp <= 70 and potions > 0:  # 신중 전략에서는 HP가 낮고 포션이 있을 때 회복합니다.
        return "heal_player"  # 회복 도구 이름을 반환합니다.
    return "attack_enemy"  # 회복할 이유가 없으면 공격 도구를 선택합니다.

assert choose_action({"player_hp": 64, "potions": 1}, "cautious") == "heal_player"  # 낮은 HP에서 신중 전략이 회복하는지 확인합니다.
assert choose_action({"player_hp": 64, "potions": 1}, "aggressive") == "attack_enemy"  # 공격 전략은 낮은 HP에서도 공격하는지 확인합니다.
print("전략 선택 로직 테스트 통과")  # 전략 단위 테스트의 성공을 출력합니다.

전략 선택 로직 테스트 통과


## 5. Node 4개 구현 — observe → decide → act → judge

`observe`는 게임 상태를 읽습니다. `decide`는 행동을 선택합니다. `act`는 MCP Tool을 실행합니다. `judge`는 **상태 변화가 확인된 뒤** 계속할지 결정합니다.

> **주의:** `decide`는 이 실습에서 **Python 규칙**입니다. 실제 LLM의 독립적 판단이 아닙니다.

In [7]:
async def observe(state: BattleState) -> dict[str, Any]:  # MCP Tool로 최신 게임 상태를 조회하는 노드입니다.
    async with Client(mcp) as client:  # 서버 객체에 메모리 내 연결을 만듭니다.
        result = await client.call_tool("get_game_status", {})  # 읽기 전용 MCP Tool을 호출합니다.
    if result.is_error or not result.structured_content or not result.structured_content.get("ok"):  # 결과가 비정상인지 확인합니다.
        return {"route": "error", "events": ["observe: 게임 조회 실패"]}  # 진행을 멈추도록 오류 상태와 로그를 반환합니다.
    game_snapshot = result.structured_content["state"]  # 서버가 돌려준 원본 게임 상태의 복사본을 가져옵니다.
    return {"observation": game_snapshot, "events": [f"observe: HP={game_snapshot['player_hp']}, 적 HP={game_snapshot['enemy_hp']}"]}  # 최신 관측값과 로그를 기록합니다.

async def decide(state: BattleState) -> dict[str, Any]:  # 게임 상황을 보고 다음 행동을 선택하는 노드입니다.
    if state.get("route") == "error":  # 앞 단계의 오류가 있는지 확인합니다.
        return {"action": "", "events": ["decide: 앞 단계 오류로 선택 생략"]}  # 오류가 있으면 행동을 선택하지 않습니다.
    game_snapshot = state["observation"]  # 방금 조회한 게임 상태를 참조합니다.
    selected_action = choose_action(game_snapshot, state["strategy"])  # 전략 규칙에 따라 사용할 MCP Tool 이름을 선택합니다.
    return {"action": selected_action, "events": [f"decide: {selected_action} 선택"]}  # 행동 선택 결과와 로그를 반환합니다.

async def act(state: BattleState) -> dict[str, Any]:  # 선택한 행동을 MCP 서버에 실행 요청하는 노드입니다.
    if state.get("route") == "error":  # 앞 단계에서 오류가 있었는지 확인합니다.
        return {"events": ["act: 앞 단계 오류로 실행 생략"]}  # 오류 시 도구를 실행하지 않습니다.
    tool_name = state["action"]  # decide 노드에서 선택한 Tool 이름을 가져옵니다.
    arguments = {"target": "slime"} if tool_name == "attack_enemy" else {}  # 공격일 때만 대상 인자를 넣습니다.
    async with Client(mcp) as client:  # 인메모리 MCP Client를 생성합니다.
        result = await client.call_tool(tool_name, arguments)  # 선택한 도구를 실제로 호출해 게임 원본을 변경합니다.
    payload = result.structured_content or {}  # 구조화된 결과가 없을 때 빈 객체를 사용합니다.
    if result.is_error or not payload.get("ok"):  # SDK 오류 또는 게임 업무 오류를 검사합니다.
        return {"route": "error", "last_result": payload, "events": [f"act: {tool_name} 실패: {payload.get('error_code', 'MCP_ERROR')}"]}  # 실패를 상태와 로그에 기록합니다.
    return {"steps": state["steps"] + 1, "last_result": payload, "events": [f"act: {tool_name} 실행 성공"]}  # 성공했을 때 행동 횟수와 결과를 기록합니다.

def judge(state: BattleState) -> dict[str, Any]:  # 반복하거나 종료할지 결정하는 노드입니다.
    if state.get("route") == "error":  # 앞 단계에서 오류가 발생했는지 검사합니다.
        return {"route": "error", "events": ["judge: 오류로 종료"]}  # 오류 상태를 유지하고 종료 경로를 선택합니다.
    current = state["last_result"]["state"]  # 방금 실행된 MCP 도구가 반환한 최신 게임 상태를 가져옵니다.
    if current["outcome"] != "playing":  # 게임 엔진에서 승리·패배·시간 초과를 기록했는지 확인합니다.
        return {"route": "finish", "observation": current, "events": [f"judge: {current['outcome']}로 종료"]}  # 종료 조건을 충족하면 반복하지 않습니다.
    if state["steps"] >= MAX_TURNS:  # 그래프 측에서도 최대 실행 횟수를 별도로 확인합니다.
        return {"route": "give_up", "observation": current, "events": ["judge: 행동 횟수 상한으로 종료"]}  # 자체 상한에 도달하면 더 실행하지 않습니다.
    return {"route": "continue", "observation": current, "events": ["judge: 다음 턴 진행"]}  # 끝나지 않았으면 관측 노드로 되돌아갑니다.

def route_after_judge(state: BattleState) -> str:  # judge 이후의 그래프 경로를 고르는 함수입니다.
    return "observe" if state["route"] == "continue" else "end"  # continue일 때만 반복하고 나머지는 종료합니다.

## 6. 그래프 연결 · 컴파일

```text
START → observe → decide → act → judge ── 종료 → END
           ↑                       │
           └──────── 계속 ←─────────┘
```

반복은 모델이 마음대로 실행하는 것이 아니라 **개발자가 명시한 조건부 Edge**에 의해 제어됩니다.

In [8]:
builder = StateGraph(BattleState)  # BattleState를 공유하는 LangGraph 설계도를 만듭니다.
builder.add_node("observe", observe)  # 현재 게임 상태 조회 노드를 등록합니다.
builder.add_node("decide", decide)  # 다음 행동 선택 노드를 등록합니다.
builder.add_node("act", act)  # 실제 MCP Tool 호출 노드를 등록합니다.
builder.add_node("judge", judge)  # 게임 종료·반복 판정 노드를 등록합니다.
builder.add_edge(START, "observe")  # 그래프 시작 후 observe로 이동합니다.
builder.add_edge("observe", "decide")  # 조회한 상태를 decide로 전달합니다.
builder.add_edge("decide", "act")  # 결정된 행동을 act로 전달합니다.
builder.add_edge("act", "judge")  # 실행 결과를 judge에서 평가하도록 연결합니다.
builder.add_conditional_edges("judge", route_after_judge, {"observe": "observe", "end": END})  # 계속이면 observe로 되돌리고 아니면 종료합니다.
app = builder.compile()  # 그래프를 실행 가능한 객체로 컴파일합니다.
print("그래프 컴파일 완료")  # 컴파일이 정상적으로 끝났는지 확인합니다.

그래프 컴파일 완료


## 7. 전투 실행 및 로그 확인

전투 전마다 게임 엔진을 재설정한 다음 `app.ainvoke()`로 그래프를 실행합니다. `recursion_limit`은 프레임워크 안전 상한이며, **게임의 최대 턴 규칙을 대체하지 않습니다**.

In [9]:
async def run_battle(strategy: str = "cautious", starting_hp: int = 100) -> BattleState:  # 새로운 전투를 시작하고 그래프를 실행하는 헬퍼 함수입니다.
    reset_result = reset_logic(starting_hp)  # MCP 서버가 관리하는 게임 상태를 처음부터 다시 시작합니다.
    assert reset_result["ok"]  # 잘못된 HP 등으로 초기화가 실패했다면 중단합니다.
    initial: BattleState = {"observation": {}, "action": "", "last_result": {}, "steps": 0, "route": "continue", "strategy": strategy, "events": []}  # 그래프에 전달할 모든 필드를 초기화합니다.
    completed = await app.ainvoke(initial, config={"recursion_limit": 50})  # 실행 상한을 명시하고 LangGraph 전투를 시작합니다.
    return completed  # 모든 노드가 갱신한 최종 State를 돌려줍니다.

battle = await run_battle("cautious")  # HP 100에서 신중 전략의 전투를 한 번 실행합니다.
for log in battle["events"]:  # 실행 중 쌓인 모든 이벤트를 순서대로 조회합니다.
    print(log)  # 각 단계에서 어떤 판단과 호출이 일어났는지 보여줍니다.
print("최종 결과:", battle["observation"])  # 마지막 HP·포션·턴·승패를 출력합니다.
assert battle["observation"]["outcome"] == "win"  # 기본 조건에서 몬스터를 이겼는지 확인합니다.
assert battle["steps"] == battle["observation"]["turn"]  # LangGraph 행동 수와 게임 서버 턴 수가 같은지 확인합니다.

observe: HP=100, 적 HP=60
decide: attack_enemy 선택
act: attack_enemy 실행 성공
judge: 다음 턴 진행
observe: HP=82, 적 HP=40
decide: attack_enemy 선택
act: attack_enemy 실행 성공
judge: 다음 턴 진행
observe: HP=64, 적 HP=20
decide: heal_player 선택
act: heal_player 실행 성공
judge: 다음 턴 진행
observe: HP=76, 적 HP=20
decide: attack_enemy 선택
act: attack_enemy 실행 성공
judge: win로 종료
최종 결과: {'player_hp': 76, 'enemy_hp': 0, 'potions': 1, 'turn': 4, 'outcome': 'win'}


## 8. 두 전략 비교 — 신중 vs 공격

`cautious`는 HP가 70 이하이고 포션이 남았다면 회복, `aggressive`는 계속 공격합니다. 같은 시작 HP에서 두 전투의 **턴 수, 남은 HP, 포션 수**를 비교합니다. 이 게임은 난수 없이 동작하므로 결과가 매번 같습니다.

In [10]:
strategies = ["cautious", "aggressive"]  # 비교할 두 전략 이름을 지정합니다.
comparison = []  # 각 전략의 최종 성적을 저장할 목록입니다.
for strategy in strategies:  # 두 전략을 하나씩 독립적으로 실행합니다.
    finished = await run_battle(strategy)  # 해당 전략으로 새 전투를 시작합니다.
    summary = finished["observation"]  # 종료 시점의 게임 상태를 꺼냅니다.
    comparison.append({"strategy": strategy, "outcome": summary["outcome"], "turns": summary["turn"], "player_hp": summary["player_hp"], "potions": summary["potions"]})  # 전략과 결과 지표를 저장합니다.
for item in comparison:  # 결과표에 저장한 두 전략을 순회합니다.
    print(item)  # 각 전략의 승패·턴·HP·포션을 출력합니다.
assert len(comparison) == 2  # 두 전략의 결과를 모두 생성했는지 확인합니다.

{'strategy': 'cautious', 'outcome': 'win', 'turns': 4, 'player_hp': 76, 'potions': 1}
{'strategy': 'aggressive', 'outcome': 'win', 'turns': 3, 'player_hp': 64, 'potions': 2}


## 9. 오류·경계·강제 종료 자동 테스트

아래는 정상 게임과 예외 상황을 함께 확인하는 **최소 계약 테스트**입니다. 특히 무한 반복을 막는 업무 상한과 잘못된 Tool 입력 차단을 분리해서 검증합니다.

In [11]:
passed = 0  # 통과한 테스트 개수를 세기 위한 변수입니다.
reset_logic()  # 다른 테스트의 상태 영향이 없도록 게임을 초기화합니다.
wrong = perform_logic("attack", "dragon")  # 허용되지 않은 몬스터 이름으로 공격을 시도합니다.
assert wrong["error_code"] == "INVALID_TARGET" and GAME.turn == 0  # 잘못된 요청은 턴을 소비하지 않아야 합니다.
passed += 1  # 첫 번째 테스트 성공을 기록합니다.
reset_logic()  # 다음 테스트를 위해 게임을 초기화합니다.
full = perform_logic("heal")  # 체력이 가득 찬 상태에서 회복을 시도합니다.
assert full["error_code"] == "HP_FULL" and GAME.potions == 2  # 포션이 소모되지 않았는지 확인합니다.
passed += 1  # 두 번째 테스트 성공을 기록합니다.
reset_logic(player_hp=50)  # 포션 사용이 가능한 HP로 게임을 시작합니다.
heal = perform_logic("heal")  # 포션을 사용해 회복합니다.
assert heal["ok"] and GAME.potions == 1 and GAME.player_hp == 62  # 50 + 30 - 18 = 62인지 검사합니다.
passed += 1  # 세 번째 테스트 성공을 기록합니다.
reset_logic()  # 공격 반복 테스트를 위해 게임을 초기화합니다.
for _ in range(3):  # 슬라임 체력 60을 소진하는 세 번의 공격을 수행합니다.
    last = perform_logic("attack")  # 공격으로 몬스터 HP를 20 낮춥니다.
assert last["state"]["outcome"] == "win"  # 세 번 공격 후 승리인지 확인합니다.
passed += 1  # 네 번째 테스트 성공을 기록합니다.
after_end = perform_logic("attack")  # 종료된 게임에서 다시 공격을 시도합니다.
assert after_end["error_code"] == "GAME_FINISHED"  # 종료 후 실행은 반드시 거부되는지 확인합니다.
passed += 1  # 다섯 번째 테스트 성공을 기록합니다.
reset_logic()  # 최대 턴 확인을 위해 게임을 초기화합니다.
for _ in range(MAX_TURNS):  # 승리 없이 끝나도록 방어 행동만 최대 턴 수만큼 실행합니다.
    last = perform_logic("defend")  # 방어로 반격 피해를 줄여 게임이 오래 지속되도록 합니다.
    if GAME.outcome != "playing":  # 도중에 플레이어가 쓰러졌는지 확인합니다.
        break  # 이미 종료되었다면 더 이상 호출하지 않습니다.
assert GAME.outcome in {"lose", "timeout"} and GAME.turn <= MAX_TURNS  # 패배나 시간 초과로 유한하게 종료되는지 확인합니다.
passed += 1  # 여섯 번째 테스트 성공을 기록합니다.
finished = await run_battle("cautious")  # 전체 LangGraph+MCP 흐름을 한 번 더 검증합니다.
assert finished["route"] == "finish" and finished["observation"]["outcome"] == "win"  # 성공적으로 그래프가 종료됐는지 확인합니다.
passed += 1  # 일곱 번째 테스트 성공을 기록합니다.
print(f"자동 테스트 {passed}개 통과")  # 통과한 테스트 수를 요약해서 보여줍니다.

자동 테스트 7개 통과


## 10. 핵심 정리

| 기술 | 이 실습에서 맡은 역할 |
|---|---|
| **Python 게임 엔진** | HP·포션·턴·승패의 실제 상태 변경 |
| **MCP Server** | 공격·회복·방어·조회 함수를 Tool로 제공 |
| **MCP Client** | 도구를 발견하고 `tools/call`로 실행 요청 |
| **LangGraph State** | 관측값, 행동, 단계 수, 로그를 노드 사이에 전달 |
| **LangGraph Edge** | `observe → decide → act → judge` 및 반복·종료 제어 |
| **LLM** | **기본 실습에는 없음**. 선택 실습에서 행동 결정을 담당할 수 있음 |

> **한 문장 정리:** MCP는 **어떤 도구를 어떻게 호출하는지**, LangGraph는 **언제 호출하고 언제 반복·종료하는지**를 제어합니다.

### 참고 자료
- LangGraph 공식 문서: https://docs.langchain.com/oss/python/langgraph/overview
- LangGraph Graph API: https://docs.langchain.com/oss/python/langgraph/graph-api
- MCP Python SDK: https://py.sdk.modelcontextprotocol.io/
